# Approach3 Multidate Window Coverage Pre-analysis

This notebook extends the single-date coverage test to a sequence of reference dates. It uses a non-overlapping reference cadence based on the selected primary window, then scans a bounded set of candidate windows around each reference date.

## Context & Methods

The default temporal logic follows the single-date test:

- Primary processing window: `+/-5` days, which is 11 inclusive calendar days.
- Reference-date cadence: every 11 days, so primary windows do not overlap.
- First default reference date: `START_DATE`, so the first window may use support observations before the analysis year.
- Window scan: `0..10` days around each reference date, used to diagnose whether a wider adaptive window would improve coverage.
- Source hierarchy: Dynamic World first, OPERA HLS Landsat only where DW is invalid, then OPERA S1 only where DW and HLS are invalid.

Outputs are saved to disk rather than displayed inline. The key review artifact is `reference_date_summary.csv` plus the PNG figures in the `figures` folder.

## 1. Parameters

Edit this cell before running. `END_DATE` is exclusive.

In [1]:
# Earth Engine
EE_PROJECT = "hardy-tenure-383607"
USE_HIGH_VOLUME_ENDPOINT = False

# AOI options: "drawn", "basin", "bbox", "point_buffer", or "geojson".
AOI_LABEL = "drawn_multidate_2025"
AOI_MODE = "drawn"
HYBAS_ID = 1041259950
HYDROBASINS_LEVEL = 4
AOI_BBOX = None  # Example: (29.0, -7.0, 30.0, -6.0)
AOI_GEOJSON_PATH = None
TEST_AOI_POINT_LON = 29.75
TEST_AOI_POINT_LAT = -6.5
TEST_AOI_BUFFER_M = 20000
MAP_CENTER = [-6.5, 29.5]  # [lat, lon]
MAP_ZOOM = 6

# Reference-date range. END_DATE is exclusive.
START_DATE = "2025-01-01"
END_DATE = "2026-01-01"

# Non-overlapping primary cadence. +/-5 gives 11 inclusive calendar days.
PRIMARY_WINDOW_DAYS = 5
FIRST_REFERENCE_MODE = "start_date"  # "start_date", "first_full_window", or "manual"
MANUAL_FIRST_REFERENCE_DATE = None  # Example: "2025-01-06"
MAX_REFERENCE_DATES = None  # Set an integer for a shorter smoke test.

# Diagnostic scan settings.
MAX_WINDOW_DAYS = 10
WINDOW_STEP_DAYS = 1
COVERAGE_TARGET_PCT = 99.0
AREA_SCALE_M = 300
REDUCE_TILE_SCALE = 4
REUSE_EXISTING_WINDOW_SCAN = True

# Source settings.
INCLUDE_HLS_SENTINEL2 = False  # Keep False: Dynamic World already uses Sentinel-2.

# Output settings. None builds a timestamped label.
OUTPUT_RUN_LABEL = None

## 2. Setup

This initializes Earth Engine, resolves the Approach3 package path, and creates the output folders.

In [2]:
from __future__ import annotations

from datetime import date, datetime, timedelta, timezone
from pathlib import Path
import json
import re
import sys

import ee
import geemap
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
if cwd.name == "Approach3":
    APPROACH3_ROOT = cwd
elif (cwd / "Approaches" / "Approach3").exists():
    APPROACH3_ROOT = cwd / "Approaches" / "Approach3"
else:
    APPROACH3_ROOT = next(parent for parent in cwd.parents if parent.name == "Approach3")

sys.path.insert(0, str(APPROACH3_ROOT / "src"))

from sw_dws1_approach3.aoi import AoiConfig, aoi_summary, basin_aoi, resolve_aoi
from sw_dws1_approach3.datasets import (
    DynamicWorldThresholds,
    OperaHlsWtrClass,
    OperaS1WtrClass,
    dynamic_world_collection,
    opera_dswx_hls_collection,
    opera_dswx_s1_collection,
)
from sw_dws1_approach3.gee_session import initialize_earth_engine
from sw_dws1_approach3.periods import validate_date_window

if EE_PROJECT == "your-google-cloud-project-id":
    raise ValueError("Set EE_PROJECT before running the notebook.")

validate_date_window(START_DATE, END_DATE)
if MAX_WINDOW_DAYS < PRIMARY_WINDOW_DAYS:
    raise ValueError("MAX_WINDOW_DAYS should be >= PRIMARY_WINDOW_DAYS.")

initialize_earth_engine(project=EE_PROJECT, use_high_volume_endpoint=USE_HIGH_VOLUME_ENDPOINT)

def safe_label(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_") or "run"

run_label = OUTPUT_RUN_LABEL or f"{safe_label(AOI_LABEL)}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = APPROACH3_ROOT / "notebooks" / "outputs" / "multidate_window_coverage_preanalysis" / run_label
CSV_DIR = OUTPUT_DIR / "csv"
FIGURE_DIR = OUTPUT_DIR / "figures"
CURVE_DIR = FIGURE_DIR / "per_reference_curves"
for folder in [OUTPUT_DIR, CSV_DIR, FIGURE_DIR, CURVE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Approach3 root:", APPROACH3_ROOT)
print("Output directory:", OUTPUT_DIR)

Approach3 root: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3
Output directory: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708


## 3. AOI Selection

If `AOI_MODE = "drawn"`, draw one polygon or rectangle on this map, then run the next cell. For repeated latitude tests, rerun the notebook with a different `AOI_LABEL` and AOI.


In [3]:
aoi_config = AoiConfig(
    mode=AOI_MODE,
    hybas_id=HYBAS_ID,
    hydrobasins_level=HYDROBASINS_LEVEL,
    bbox=AOI_BBOX,
    point_lon=TEST_AOI_POINT_LON,
    point_lat=TEST_AOI_POINT_LAT,
    point_buffer_m=TEST_AOI_BUFFER_M,
    geojson_path=AOI_GEOJSON_PATH,
)

reference_aoi = basin_aoi(HYBAS_ID, level=HYDROBASINS_LEVEL) if AOI_MODE == "drawn" else resolve_aoi(aoi_config)

draw_map = geemap.Map(center=MAP_CENTER, zoom=MAP_ZOOM, ee_initialize=False)
draw_map.addLayer(reference_aoi, {"color": "cyan"}, "Configured/reference AOI")
draw_map.addLayerControl()
draw_map


Map(center=[-6.5, 29.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

## 4. Confirm AOI

This saves the AOI geometry used by the scan to `aoi.geojson` inside the output run folder.


In [4]:
drawn_geometry = None
if AOI_MODE == "drawn":
    if draw_map.draw_last_feature is None:
        raise ValueError("AOI_MODE is 'drawn', but no geometry has been drawn on the map above.")
    drawn_geometry = draw_map.draw_last_feature.geometry()

aoi = resolve_aoi(aoi_config, custom_geometry=drawn_geometry)
aoi_info = aoi_summary(aoi).getInfo()
aoi_area_km2 = float(aoi_info["area_km2"])

feature_geojson = {
    "type": "Feature",
    "properties": {"aoi_label": AOI_LABEL, "aoi_mode": AOI_MODE},
    "geometry": aoi.getInfo(),
}
with (OUTPUT_DIR / "aoi.geojson").open("w", encoding="utf-8") as f:
    json.dump(feature_geojson, f, indent=2)

print("AOI summary:", aoi_info)
print(f"AOI area km2: {aoi_area_km2:,.2f}")


AOI summary: {'area_km2': 1009532.7427023456, 'bounds': [[[25.57671, -9.9117], [34.563473, -9.9117], [34.563473, -0.776916], [25.57671, -0.776916], [25.57671, -9.9117]]]}
AOI area km2: 1,009,532.74


## 5. Reference Dates

With `PRIMARY_WINDOW_DAYS = 5`, references are spaced every 11 days: `2025-01-01`, `2025-01-12`, `2025-01-23`, and so on. The primary windows are non-overlapping.

In [5]:
def _parse_date(value: str) -> date:
    return date.fromisoformat(value)


def first_reference_date() -> date:
    start = _parse_date(START_DATE)
    if FIRST_REFERENCE_MODE == "start_date":
        return start
    if FIRST_REFERENCE_MODE == "first_full_window":
        return start + timedelta(days=PRIMARY_WINDOW_DAYS)
    if FIRST_REFERENCE_MODE == "manual":
        if MANUAL_FIRST_REFERENCE_DATE is None:
            raise ValueError("MANUAL_FIRST_REFERENCE_DATE is required when FIRST_REFERENCE_MODE='manual'.")
        manual = _parse_date(MANUAL_FIRST_REFERENCE_DATE)
        if not (start <= manual < _parse_date(END_DATE)):
            raise ValueError("MANUAL_FIRST_REFERENCE_DATE must be inside [START_DATE, END_DATE).")
        return manual
    raise ValueError('FIRST_REFERENCE_MODE must be "start_date", "first_full_window", or "manual".')


def build_reference_dates() -> list[str]:
    stride_days = 2 * PRIMARY_WINDOW_DAYS + 1
    end = _parse_date(END_DATE)
    current = first_reference_date()
    out = []
    while current < end:
        out.append(current.isoformat())
        if MAX_REFERENCE_DATES is not None and len(out) >= int(MAX_REFERENCE_DATES):
            break
        current += timedelta(days=stride_days)
    return out


def primary_window_bounds(target_date: str) -> tuple[str, str, str]:
    target = _parse_date(target_date)
    start = target - timedelta(days=PRIMARY_WINDOW_DAYS)
    end_exclusive = target + timedelta(days=PRIMARY_WINDOW_DAYS + 1)
    end_inclusive = end_exclusive - timedelta(days=1)
    return start.isoformat(), end_inclusive.isoformat(), end_exclusive.isoformat()


reference_dates = build_reference_dates()
if not reference_dates:
    raise ValueError("No reference dates selected.")

reference_rows = []
previous_end_inclusive = None
for target_date in reference_dates:
    start, end_inclusive, end_exclusive = primary_window_bounds(target_date)
    overlaps_previous = False
    if previous_end_inclusive is not None:
        overlaps_previous = _parse_date(start) <= _parse_date(previous_end_inclusive)
    previous_end_inclusive = end_inclusive
    reference_rows.append({
        "target_date": target_date,
        "primary_window_days": PRIMARY_WINDOW_DAYS,
        "cadence_stride_days": 2 * PRIMARY_WINDOW_DAYS + 1,
        "primary_window_start": start,
        "primary_window_end_inclusive": end_inclusive,
        "primary_window_end_exclusive": end_exclusive,
        "primary_window_calendar_days": (_parse_date(end_exclusive) - _parse_date(start)).days,
        "primary_overlaps_previous": overlaps_previous,
        "uses_support_before_analysis_start": _parse_date(start) < _parse_date(START_DATE),
        "uses_support_after_analysis_end": _parse_date(end_exclusive) > _parse_date(END_DATE),
    })

reference_df = pd.DataFrame(reference_rows)
reference_csv = CSV_DIR / "reference_windows.csv"
reference_df.to_csv(reference_csv, index=False)

print(f"Reference dates: {len(reference_dates)}")
print("Saved:", reference_csv)
display(reference_df.head(20))

Reference dates: 34
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\csv\reference_windows.csv


,target_date,primary_window_days,cadence_stride_days,primary_window_start,primary_window_end_inclusive,primary_window_end_exclusive,primary_window_calendar_days,primary_overlaps_previous,uses_support_before_analysis_start,uses_support_after_analysis_end
0,2025-01-01,5,11,2024-12-27,2025-01-06,2025-01-07,11,False,True,False
1,2025-01-12,5,11,2025-01-07,2025-01-17,2025-01-18,11,False,False,False
2,2025-01-23,5,11,2025-01-18,2025-01-28,2025-01-29,11,False,False,False
3,2025-02-03,5,11,2025-01-29,2025-02-08,2025-02-09,11,False,False,False
4,2025-02-14,5,11,2025-02-09,2025-02-19,2025-02-20,11,False,False,False
5,2025-02-25,5,11,2025-02-20,2025-03-02,2025-03-03,11,False,False,False
6,2025-03-08,5,11,2025-03-03,2025-03-13,2025-03-14,11,False,False,False
7,2025-03-19,5,11,2025-03-14,2025-03-24,2025-03-25,11,False,False,False
8,2025-03-30,5,11,2025-03-25,2025-04-04,2025-04-05,11,False,False,False
9,2025-04-10,5,11,2025-04-05,2025-04-15,2025-04-16,11,False,False,False


## 6. Coverage Helpers

These are the same validity rules used in the single-date notebook.

In [6]:
thresholds = DynamicWorldThresholds()


def dw_valid_observation(image: ee.Image) -> ee.Image:
    image = ee.Image(image)
    water = image.select("water")
    flooded = image.select("flooded_vegetation")
    valid = water.gt(thresholds.water).Or(water.lte(thresholds.nonwater)).Or(
        flooded.gt(thresholds.flooded_vegetation)
    )
    return valid.rename("valid").toByte().updateMask(water.mask())


def hls_valid_observation(image: ee.Image) -> ee.Image:
    wtr = ee.Image(image).select("WTR_Water_classification")
    return wtr.lt(OperaHlsWtrClass.SNOW_ICE).rename("valid").toByte().updateMask(wtr.mask())


def s1_valid_observation(image: ee.Image) -> ee.Image:
    wtr = ee.Image(image).select("WTR_Water_classification")
    return wtr.lt(OperaS1WtrClass.HAND_MASKED).rename("valid").toByte().updateMask(wtr.mask())


def empty_valid_image() -> ee.Image:
    return ee.Image.constant(0).rename("valid").toByte().clip(aoi)


def any_valid_mask(collection: ee.ImageCollection, valid_fn) -> ee.Image:
    valid_collection = ee.ImageCollection(
        collection.map(lambda image: valid_fn(ee.Image(image)).unmask(0))
    ).merge(ee.ImageCollection([empty_valid_image()]))
    return valid_collection.sum().gt(0).rename("valid_any").clip(aoi)


def window_bounds(target_date: str, window_days: int) -> tuple[str, str]:
    target = _parse_date(target_date)
    start = target - timedelta(days=window_days)
    end = target + timedelta(days=window_days + 1)
    return start.isoformat(), end.isoformat()


def build_window_sources(target_date: str, window_days: int) -> dict:
    start, end = window_bounds(target_date, window_days)
    dw_collection = dynamic_world_collection(aoi, start, end)
    hls_collection = opera_dswx_hls_collection(
        aoi,
        start,
        end,
        include_sentinel2=INCLUDE_HLS_SENTINEL2,
    )
    s1_collection = opera_dswx_s1_collection(aoi, start, end)

    dw_valid = any_valid_mask(dw_collection, dw_valid_observation).unmask(0).eq(1)
    hls_valid = any_valid_mask(hls_collection, hls_valid_observation).unmask(0).eq(1)
    s1_valid = any_valid_mask(s1_collection, s1_valid_observation).unmask(0).eq(1)

    dw_gap = dw_valid.Not()
    hls_used = dw_gap.And(hls_valid)
    after_hls_gap = dw_gap.And(hls_valid.Not())
    s1_used = after_hls_gap.And(s1_valid)
    final_valid = dw_valid.Or(hls_used).Or(s1_used)
    remaining_gap = final_valid.Not()

    return {
        "target_date": target_date,
        "window_days": window_days,
        "window_start": start,
        "window_end_inclusive": (_parse_date(end) - timedelta(days=1)).isoformat(),
        "window_end_exclusive": end,
        "collections": {"dw": dw_collection, "hls": hls_collection, "s1": s1_collection},
        "masks": {
            "dw_valid": dw_valid,
            "hls_valid": hls_valid,
            "s1_valid": s1_valid,
            "dw_gap": dw_gap,
            "hls_used": hls_used,
            "after_hls_gap": after_hls_gap,
            "s1_used": s1_used,
            "final_valid": final_valid,
            "remaining_gap": remaining_gap,
        },
    }


def area_stats(mask_dict: dict[str, ee.Image]) -> dict[str, float]:
    area_bands = []
    for name, mask in mask_dict.items():
        area_bands.append(ee.Image.pixelArea().rename(name).updateMask(mask))
    stats = ee.Image.cat(area_bands).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=AREA_SCALE_M,
        maxPixels=1e13,
        tileScale=REDUCE_TILE_SCALE,
    ).getInfo()
    return {name: (stats.get(name) or 0) / 1_000_000 for name in mask_dict}


def pct(area_km2: float, denominator_km2: float) -> float:
    if denominator_km2 <= 0:
        return 0.0
    return 100.0 * area_km2 / denominator_km2


def scan_target_window(target_date: str, window_days: int) -> dict:
    sources = build_window_sources(target_date, window_days)
    masks = sources["masks"]
    areas = area_stats(masks)
    counts = ee.Dictionary({
        "dw_image_count": sources["collections"]["dw"].size(),
        "hls_image_count": sources["collections"]["hls"].size(),
        "s1_image_count": sources["collections"]["s1"].size(),
    }).getInfo()

    dw_gap_area = areas["dw_gap"]
    after_hls_gap_area = areas["after_hls_gap"]

    return {
        "aoi_label": AOI_LABEL,
        "target_date": target_date,
        "window_days": window_days,
        "window_start": sources["window_start"],
        "window_end_inclusive": sources["window_end_inclusive"],
        "window_end_exclusive": sources["window_end_exclusive"],
        "window_calendar_days": (_parse_date(sources["window_end_exclusive"]) - _parse_date(sources["window_start"])).days,
        **counts,
        "dw_valid_pct": pct(areas["dw_valid"], aoi_area_km2),
        "hls_valid_pct": pct(areas["hls_valid"], aoi_area_km2),
        "s1_valid_pct": pct(areas["s1_valid"], aoi_area_km2),
        "hls_used_pct_aoi": pct(areas["hls_used"], aoi_area_km2),
        "s1_used_pct_aoi": pct(areas["s1_used"], aoi_area_km2),
        "final_valid_pct": pct(areas["final_valid"], aoi_area_km2),
        "remaining_gap_pct": pct(areas["remaining_gap"], aoi_area_km2),
        "hls_fills_dw_gap_pct": pct(areas["hls_used"], dw_gap_area),
        "s1_fills_after_hls_gap_pct": pct(areas["s1_used"], after_hls_gap_area),
        "dw_valid_km2": areas["dw_valid"],
        "hls_used_km2": areas["hls_used"],
        "s1_used_km2": areas["s1_used"],
        "final_valid_km2": areas["final_valid"],
        "remaining_gap_km2": areas["remaining_gap"],
    }

## 7. Run Window Scan

This scans each reference date for windows from `0` to `MAX_WINDOW_DAYS`. Results are saved incrementally, so if the notebook stops you can rerun this cell with `REUSE_EXISTING_WINDOW_SCAN = True`.

In [7]:
window_days_list = list(range(0, MAX_WINDOW_DAYS + 1, WINDOW_STEP_DAYS))
window_csv = CSV_DIR / "window_scan_results.csv"

existing_rows = []
completed_keys = set()
if REUSE_EXISTING_WINDOW_SCAN and window_csv.exists():
    existing_df = pd.read_csv(window_csv)
    required_cols = {"target_date", "window_days"}
    if required_cols.issubset(existing_df.columns):
        existing_rows = existing_df.to_dict("records")
        completed_keys = {
            (str(row["target_date"]), int(row["window_days"]))
            for row in existing_rows
        }
        print(f"Loaded {len(existing_rows)} existing scan rows from {window_csv}")

scan_plan = [
    (target_date, window_days)
    for target_date in reference_dates
    for window_days in window_days_list
]
remaining_plan = [key for key in scan_plan if key not in completed_keys]
print(f"Scan plan: {len(reference_dates)} reference dates x {len(window_days_list)} windows = {len(scan_plan)} rows")
print(f"Remaining rows to compute: {len(remaining_plan)}")

rows = list(existing_rows)
for i, (target_date, window_days) in enumerate(remaining_plan, start=1):
    print(f"[{i}/{len(remaining_plan)}] target={target_date}, window=+-{window_days} days")
    rows.append(scan_target_window(target_date, window_days))
    pd.DataFrame(rows).sort_values(["target_date", "window_days"]).to_csv(window_csv, index=False)

window_df = pd.DataFrame(rows).sort_values(["target_date", "window_days"]).reset_index(drop=True)
window_df.to_csv(window_csv, index=False)

print("Saved:", window_csv)
display(window_df.head(20))

Scan plan: 34 reference dates x 11 windows = 374 rows
Remaining rows to compute: 374
[1/374] target=2025-01-01, window=+-0 days
[2/374] target=2025-01-01, window=+-1 days
[3/374] target=2025-01-01, window=+-2 days
[4/374] target=2025-01-01, window=+-3 days
[5/374] target=2025-01-01, window=+-4 days
[6/374] target=2025-01-01, window=+-5 days
[7/374] target=2025-01-01, window=+-6 days
[8/374] target=2025-01-01, window=+-7 days
[9/374] target=2025-01-01, window=+-8 days
[10/374] target=2025-01-01, window=+-9 days
[11/374] target=2025-01-01, window=+-10 days
[12/374] target=2025-01-12, window=+-0 days
[13/374] target=2025-01-12, window=+-1 days
[14/374] target=2025-01-12, window=+-2 days
[15/374] target=2025-01-12, window=+-3 days
[16/374] target=2025-01-12, window=+-4 days
[17/374] target=2025-01-12, window=+-5 days
[18/374] target=2025-01-12, window=+-6 days
[19/374] target=2025-01-12, window=+-7 days
[20/374] target=2025-01-12, window=+-8 days
[21/374] target=2025-01-12, window=+-9 days

,aoi_label,target_date,window_days,window_start,window_end_inclusive,window_end_exclusive,window_calendar_days,dw_image_count,hls_image_count,s1_image_count,...,s1_used_pct_aoi,final_valid_pct,remaining_gap_pct,hls_fills_dw_gap_pct,s1_fills_after_hls_gap_pct,dw_valid_km2,hls_used_km2,s1_used_km2,final_valid_km2,remaining_gap_km2
0,drawn_multidate_2025,2025-01-01,0,2025-01-01,2025-01-01,2025-01-02,1,3,49,50,...,24.617549,27.181825,72.386034,2.467152,25.377979,1115.650871,24771.546062,248522.222132,2.744094e+05,730760.716485
1,drawn_multidate_2025,2025-01-01,1,2024-12-31,2025-01-02,2025-01-03,3,16,77,50,...,24.617549,30.762753,68.805106,3.226152,26.350728,30596.620768,31441.220783,248522.222132,3.105601e+05,694610.071866
2,drawn_multidate_2025,2025-01-01,2,2024-12-30,2025-01-03,2025-01-04,5,47,190,135,...,53.017078,72.723190,26.844669,11.500867,66.386074,94166.312944,104773.335859,535224.765833,7.341644e+05,271005.720914
3,drawn_multidate_2025,2025-01-01,3,2024-12-29,2025-01-04,2025-01-05,7,76,282,193,...,50.880209,83.502566,16.065293,19.878082,76.002431,161659.674620,167673.699799,513652.371280,8.429857e+05,162184.389852
4,drawn_multidate_2025,2025-01-01,4,2024-12-28,2025-01-05,2025-01-06,9,95,341,193,...,49.550217,84.479735,15.088124,19.923545,76.657625,190267.408325,162357.513064,500225.661570,8.528506e+05,152319.552592
5,drawn_multidate_2025,2025-01-01,5,2024-12-27,2025-01-06,2025-01-07,11,147,434,271,...,53.045612,99.354742,0.213117,26.318116,99.599846,275459.824361,192046.009573,535512.817599,1.003019e+06,2151.484017
6,drawn_multidate_2025,2025-01-01,6,2024-12-26,2025-01-07,2025-01-08,13,181,490,271,...,46.412979,99.374932,0.192927,33.287391,99.586046,299903.336424,234764.920525,468554.216158,1.003222e+06,1947.662444
7,drawn_multidate_2025,2025-01-01,7,2024-12-25,2025-01-08,2025-01-09,15,211,569,349,...,42.990912,99.380166,0.187693,37.012527,99.565310,313124.329969,256143.643677,434007.336845,1.003275e+06,1894.825059
8,drawn_multidate_2025,2025-01-01,8,2024-12-24,2025-01-09,2025-01-10,17,254,653,349,...,36.581037,99.394901,0.172958,44.694494,99.529418,334271.995856,299854.531408,369297.543921,1.003424e+06,1746.064366
9,drawn_multidate_2025,2025-01-01,9,2024-12-23,2025-01-10,2025-01-11,19,278,717,407,...,35.345160,99.401850,0.166009,45.592238,99.532516,346262.533711,300410.725143,356820.962234,1.003494e+06,1675.914461


## 8. Build Summary Tables

The fixed primary-window table is the non-overlapping processing candidate. The optimal table is diagnostic: it tells us the minimum window needed to reach the coverage target, up to the scan limit.

In [9]:
if "window_df" not in globals():
    if window_csv.exists():
        window_df = pd.read_csv(window_csv)
    else:
        raise RuntimeError("Run the window scan first, or point OUTPUT_DIR to a folder with window_scan_results.csv.")


def select_optimal_window(group: pd.DataFrame) -> pd.Series:
    ordered = group.sort_values(["window_days", "remaining_gap_pct"])
    eligible = ordered[ordered["final_valid_pct"] >= COVERAGE_TARGET_PCT]
    if not eligible.empty:
        row = eligible.iloc[0].copy()
        row["meets_target"] = True
        return row
    best = group.sort_values(["final_valid_pct", "window_days"], ascending=[False, True]).iloc[0].copy()
    best["meets_target"] = False
    return best


optimal_rows = []
for target_date, group in window_df.groupby("target_date", sort=True):
    row = select_optimal_window(group).copy()
    row["target_date"] = target_date
    optimal_rows.append(row)
optimal_df = pd.DataFrame(optimal_rows).reset_index(drop=True)

primary_df = window_df[window_df["window_days"] == PRIMARY_WINDOW_DAYS].copy().reset_index(drop=True)
if len(primary_df) != len(reference_dates):
    missing = sorted(set(reference_dates) - set(primary_df["target_date"].astype(str)))
    raise RuntimeError(f"Primary window rows are missing for: {missing[:10]}")

reference_df = pd.read_csv(CSV_DIR / "reference_windows.csv")
reference_df = reference_df.rename(
    columns={
        column: f"reference_{column}"
        for column in reference_df.columns
        if column.startswith("primary_")
    }
)
summary_df = reference_df.merge(
    primary_df.add_prefix("primary_"),
    left_on="target_date",
    right_on="primary_target_date",
    how="left",
).merge(
    optimal_df.add_prefix("optimal_"),
    left_on="target_date",
    right_on="optimal_target_date",
    how="left",
)

summary_df["primary_meets_target"] = summary_df["primary_final_valid_pct"] >= COVERAGE_TARGET_PCT
summary_df["adaptive_window_needed"] = summary_df["optimal_window_days"] > PRIMARY_WINDOW_DAYS
summary_df["optimal_window_exceeds_primary_cadence"] = summary_df["optimal_window_days"] > PRIMARY_WINDOW_DAYS
summary_df["primary_gap_km2_per_pct_point"] = aoi_area_km2 / 100.0

optimal_csv = CSV_DIR / "optimal_windows.csv"
primary_csv = CSV_DIR / "fixed_primary_windows.csv"
summary_csv = CSV_DIR / "reference_date_summary.csv"
optimal_df.to_csv(optimal_csv, index=False)
primary_df.to_csv(primary_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

print("Saved:", optimal_csv)
print("Saved:", primary_csv)
print("Saved:", summary_csv)

summary_cols = [
    "target_date",
    "primary_window_start",
    "primary_window_end_inclusive",
    "primary_final_valid_pct",
    "primary_remaining_gap_pct",
    "primary_dw_valid_pct",
    "primary_hls_used_pct_aoi",
    "primary_s1_used_pct_aoi",
    "primary_meets_target",
    "optimal_window_days",
    "optimal_final_valid_pct",
    "adaptive_window_needed",
]
missing_summary_cols = [column for column in summary_cols if column not in summary_df.columns]
if missing_summary_cols:
    print("Missing display columns:", missing_summary_cols)
    print("Available columns:", list(summary_df.columns))
    display(summary_df.head(30))
else:
    display(summary_df[summary_cols].head(30))

Task was destroyed but it is pending!
task: <Task pending name='Task-1060' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\ibana\AppData\Roaming\Python\Python311\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-1061' coro=<Kernel.shell_main() running at C:\Users\ibana\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\ibana\AppData\Roaming\Python\Python311\site-packages\zmq\eventloop\zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-1061' coro=<Kernel.shell_main() running at C:\Users\ibana\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>


Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\csv\optimal_windows.csv
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\csv\fixed_primary_windows.csv
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\csv\reference_date_summary.csv


,target_date,primary_window_start,primary_window_end_inclusive,primary_final_valid_pct,primary_remaining_gap_pct,primary_dw_valid_pct,primary_hls_used_pct_aoi,primary_s1_used_pct_aoi,primary_meets_target,optimal_window_days,optimal_final_valid_pct,adaptive_window_needed
0,2025-01-01,2024-12-27,2025-01-06,99.354742,0.213117,27.285873,19.023257,53.045612,True,5,99.354742,False
1,2025-01-12,2025-01-07,2025-01-17,90.412144,9.155715,25.032846,28.672471,36.706827,False,6,99.392121,True
2,2025-01-23,2025-01-18,2025-01-28,99.290079,0.277780,7.765526,24.870297,66.654255,True,5,99.290079,False
3,2025-02-03,2025-01-29,2025-02-08,96.997509,2.570350,10.611853,11.909683,74.475972,False,6,99.242929,True
4,2025-02-14,2025-02-09,2025-02-19,99.350163,0.217696,46.240485,17.802570,35.307108,True,5,99.350163,False
5,2025-02-25,2025-02-20,2025-03-02,99.495314,0.072545,69.617282,18.291022,11.587011,True,5,99.495314,False
6,2025-03-08,2025-03-03,2025-03-13,98.500137,1.067722,31.518317,34.157637,32.824183,False,6,99.378229,True
7,2025-03-19,2025-03-14,2025-03-24,99.378134,0.189724,32.133736,25.308475,41.935924,True,5,99.378134,False
8,2025-03-30,2025-03-25,2025-04-04,91.043505,8.524353,34.097184,21.113764,35.832558,False,6,99.303737,True
9,2025-04-10,2025-04-05,2025-04-15,99.460601,0.107258,43.491606,38.633954,17.335041,True,5,99.460601,False


## 9. Save PNG Figures

Figures are saved as PNG files and are not displayed inline. Color meanings are consistent across figures:

- Blue: Dynamic World valid coverage.
- Orange: OPERA HLS Landsat contribution where DW is invalid.
- Purple: OPERA S1 contribution where DW and HLS are invalid.
- Red: remaining AOI gap.
- Black: final valid coverage line.
- Gray dashed line: target coverage.

In [10]:
from PIL import Image, ImageDraw, ImageFont

COLORS = {
    "dw": "#419bdf",
    "hls": "#e49635",
    "s1": "#7a87c6",
    "gap": "#d73027",
    "final": "#111111",
    "target": "#666666",
    "grid": "#dddddd",
    "axis": "#555555",
    "text": "#222222",
    "muted": "#666666",
    "bg": "#ffffff",
}


def rgb(value: str) -> tuple[int, int, int]:
    value = value.lstrip("#")
    return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))


def get_font(size: int, bold: bool = False):
    candidates = [
        "C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf",
        "C:/Windows/Fonts/calibrib.ttf" if bold else "C:/Windows/Fonts/calibri.ttf",
    ]
    for candidate in candidates:
        try:
            return ImageFont.truetype(candidate, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_TITLE = get_font(20, True)
FONT_LABEL = get_font(12)
FONT_SMALL = get_font(10)


def text_w(draw, text, font=FONT_SMALL):
    box = draw.textbbox((0, 0), str(text), font=font)
    return box[2] - box[0]


def dashed_hline(draw, x1, x2, y, color, width=1, dash=6, gap=4):
    x = x1
    while x < x2:
        draw.line((x, y, min(x + dash, x2), y), fill=rgb(color), width=width)
        x += dash + gap


def draw_legend(draw, x, y, entries):
    cursor = x
    for label, color in entries:
        draw.rectangle((cursor, y + 3, cursor + 12, y + 15), fill=rgb(color))
        draw.text((cursor + 17, y), label, fill=rgb(COLORS["text"]), font=FONT_LABEL)
        cursor += 30 + text_w(draw, label, FONT_LABEL)


def base_canvas(width, height, title, legend_entries=None):
    image = Image.new("RGB", (width, height), rgb(COLORS["bg"]))
    draw = ImageDraw.Draw(image)
    draw.text((70, 22), title, fill=rgb(COLORS["text"]), font=FONT_TITLE)
    if legend_entries:
        draw_legend(draw, 70, 54, legend_entries)
    return image, draw


def save_stacked_source_plot(df: pd.DataFrame, path: Path, title: str, prefix: str = ""):
    data = df.sort_values("target_date").reset_index(drop=True)
    n = len(data)
    width = max(1200, 170 + n * 34)
    height = 560
    left, right, top, bottom = 75, 35, 92, 90
    plot_w = width - left - right
    plot_h = height - top - bottom
    image, draw = base_canvas(width, height, title, [
        ("Dynamic World", COLORS["dw"]),
        ("HLS Landsat fill", COLORS["hls"]),
        ("S1 fill", COLORS["s1"]),
        ("Remaining gap", COLORS["gap"]),
    ])

    def sy(value):
        return top + (100.0 - float(value)) / 100.0 * plot_h

    for tick in [0, 25, 50, 75, 100]:
        y = sy(tick)
        draw.line((left, y, left + plot_w, y), fill=rgb(COLORS["grid"]))
        draw.text((22, y - 7), f"{tick}%", fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    draw.line((left, top, left, top + plot_h), fill=rgb(COLORS["axis"]))
    draw.line((left, top + plot_h, left + plot_w, top + plot_h), fill=rgb(COLORS["axis"]))

    bar_w = max(8, min(24, plot_w / max(n, 1) * 0.65))
    step = (plot_w - bar_w) / max(n - 1, 1)
    label_every = max(1, round(n / 12))
    for idx, row in data.iterrows():
        x0 = left + idx * step
        x1 = x0 + bar_w
        y_base = top + plot_h
        cumulative = 0.0
        for column, color in [
            (f"{prefix}dw_valid_pct", COLORS["dw"]),
            (f"{prefix}hls_used_pct_aoi", COLORS["hls"]),
            (f"{prefix}s1_used_pct_aoi", COLORS["s1"]),
            (f"{prefix}remaining_gap_pct", COLORS["gap"]),
        ]:
            cumulative += float(row[column])
            y_top = sy(cumulative)
            draw.rectangle((x0, y_top, x1, y_base), fill=rgb(color))
            y_base = y_top
        if idx % label_every == 0 or idx == n - 1:
            label = str(row["target_date"])[5:]
            draw.text((x0 + bar_w / 2 - text_w(draw, label) / 2, top + plot_h + 10), label, fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    draw.text((left, height - 30), "Reference labels show MM-DD. Bars sum to 100% of AOI.", fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    image.save(path)


def save_line_plot(df: pd.DataFrame, path: Path, title: str, series, y_max=100.0, target_line=None, target_label=None, x_col=None, x_label=None):
    data = df.sort_values(x_col or "target_date").reset_index(drop=True)
    n = len(data)
    width = 900 if x_col == "window_days" else max(1200, 170 + n * 34)
    height = 500
    left, right, top, bottom = 75, 35, 92, 95
    plot_w = width - left - right
    plot_h = height - top - bottom
    image, draw = base_canvas(width, height, title, [(label, color) for _, label, color in series])

    if x_col == "window_days":
        x_values = [float(v) for v in data[x_col]]
        x_min, x_max = min(x_values), max(x_values)
        if x_max == x_min:
            x_max = x_min + 1
        sx = lambda v: left + (float(v) - x_min) / (x_max - x_min) * plot_w
        tick_positions = list(zip(x_values, [str(int(v)) for v in x_values]))
    else:
        sx = lambda i: left + i / max(n - 1, 1) * plot_w
        step = max(1, round(n / 12))
        tick_positions = [(i, str(data.loc[i, "target_date"])[5:]) for i in range(0, n, step)]
        if n - 1 not in [i for i, _ in tick_positions]:
            tick_positions.append((n - 1, str(data.loc[n - 1, "target_date"])[5:]))

    def sy(value):
        return top + (float(y_max) - float(value)) / max(float(y_max), 1.0) * plot_h

    for tick in [0, y_max * 0.25, y_max * 0.5, y_max * 0.75, y_max]:
        y = sy(tick)
        draw.line((left, y, left + plot_w, y), fill=rgb(COLORS["grid"]))
        draw.text((20, y - 7), f"{tick:g}", fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    if target_line is not None:
        y = sy(target_line)
        dashed_hline(draw, left, left + plot_w, y, COLORS["target"])
        label = target_label or f"target {target_line:g}%"
        draw.text((left + plot_w - text_w(draw, label) - 4, y - 18), label, fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    draw.line((left, top, left, top + plot_h), fill=rgb(COLORS["axis"]))
    draw.line((left, top + plot_h, left + plot_w, top + plot_h), fill=rgb(COLORS["axis"]))

    for column, _label, color in series:
        points = []
        for idx, row in data.iterrows():
            xval = row[x_col] if x_col else idx
            points.append((sx(xval), sy(row[column])))
        if len(points) > 1:
            draw.line(points, fill=rgb(color), width=3)
        for x, y in points:
            draw.ellipse((x - 3, y - 3, x + 3, y + 3), fill=rgb(color))

    for xpos, label in tick_positions:
        x = sx(xpos)
        draw.text((x - text_w(draw, label) / 2, top + plot_h + 10), label, fill=rgb(COLORS["muted"]), font=FONT_SMALL)
    if x_label:
        draw.text((left + plot_w / 2 - text_w(draw, x_label, FONT_LABEL) / 2, height - 34), x_label, fill=rgb(COLORS["muted"]), font=FONT_LABEL)
    image.save(path)


figure_manifest = []


def register(path: Path, description: str):
    figure_manifest.append({"path": str(path), "description": description})
    print("Saved:", path)


fixed_source_png = FIGURE_DIR / "summary_fixed_window_source_contribution.png"
save_stacked_source_plot(summary_df, fixed_source_png, f"Source contribution by reference date, fixed +/-{PRIMARY_WINDOW_DAYS} days", prefix="primary_")
register(fixed_source_png, "Stacked AOI percentage for the non-overlapping fixed primary window.")

optimal_source_png = FIGURE_DIR / "summary_optimal_window_source_contribution.png"
save_stacked_source_plot(optimal_df, optimal_source_png, "Source contribution by reference date, selected optimal window", prefix="")
register(optimal_source_png, "Stacked AOI percentage using the minimum window that meets the target, when available.")

coverage_comparison_png = FIGURE_DIR / "summary_final_coverage_comparison.png"
save_line_plot(
    summary_df,
    coverage_comparison_png,
    "Final AOI coverage: fixed primary window vs selected optimal window",
    [("primary_final_valid_pct", f"Fixed +/-{PRIMARY_WINDOW_DAYS}", COLORS["final"]), ("optimal_final_valid_pct", "Optimal", COLORS["hls"])],
    y_max=100,
    target_line=COVERAGE_TARGET_PCT,
)
register(coverage_comparison_png, "Final valid AOI percentage for fixed and optimal windows.")

window_days_png = FIGURE_DIR / "summary_optimal_window_days.png"
save_line_plot(
    optimal_df,
    window_days_png,
    "Minimum +/- day window needed to meet target",
    [("window_days", "selected window days", COLORS["hls"])],
    y_max=MAX_WINDOW_DAYS,
    target_line=PRIMARY_WINDOW_DAYS,
    target_label=f"primary +/-{PRIMARY_WINDOW_DAYS}",
)
register(window_days_png, "Minimum +/- day window needed for each reference date.")

image_counts_png = FIGURE_DIR / "summary_image_counts_fixed_window.png"
save_line_plot(
    primary_df,
    image_counts_png,
    f"Image counts inside fixed +/-{PRIMARY_WINDOW_DAYS} day windows",
    [("dw_image_count", "DW count", COLORS["dw"]), ("hls_image_count", "HLS count", COLORS["hls"]), ("s1_image_count", "S1 count", COLORS["s1"])],
    y_max=max(1, float(primary_df[["dw_image_count", "hls_image_count", "s1_image_count"]].max().max())),
)
register(image_counts_png, "Source image counts in the fixed primary windows.")

curve_rows = []
for target_date, group in window_df.groupby("target_date", sort=True):
    curve_path = CURVE_DIR / f"coverage_curve_{str(target_date).replace('-', '_')}.png"
    save_line_plot(
        group,
        curve_path,
        f"Coverage scan: {target_date}",
        [("dw_valid_pct", "DW valid", COLORS["dw"]), ("hls_used_pct_aoi", "HLS fill", COLORS["hls"]), ("s1_used_pct_aoi", "S1 fill", COLORS["s1"]), ("final_valid_pct", "Final valid", COLORS["final"])],
        y_max=100,
        target_line=COVERAGE_TARGET_PCT,
        x_col="window_days",
        x_label="+/- window days",
    )
    curve_rows.append({"target_date": target_date, "path": str(curve_path)})

coverage_curve_index = CSV_DIR / "coverage_curve_png_index.csv"
pd.DataFrame(curve_rows).to_csv(coverage_curve_index, index=False)
print("Saved:", coverage_curve_index)

figure_manifest_csv = CSV_DIR / "figure_manifest.csv"
pd.DataFrame(figure_manifest).to_csv(figure_manifest_csv, index=False)
print("Saved:", figure_manifest_csv)

Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\figures\summary_fixed_window_source_contribution.png
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\figures\summary_optimal_window_source_contribution.png
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\figures\summary_final_coverage_comparison.png
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\figures\summary_optimal_window_days.png
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\figures\summary_image_counts_fixed_window.png


## 10. Save Parameters And Output Index

This records the settings used for the run and builds a compact output index.

In [11]:
parameters = {
    "aoi_label": AOI_LABEL,
    "aoi_mode": AOI_MODE,
    "hybas_id": HYBAS_ID,
    "hydrobasins_level": HYDROBASINS_LEVEL,
    "start_date": START_DATE,
    "end_date_exclusive": END_DATE,
    "primary_window_days": PRIMARY_WINDOW_DAYS,
    "first_reference_mode": FIRST_REFERENCE_MODE,
    "manual_first_reference_date": MANUAL_FIRST_REFERENCE_DATE,
    "max_reference_dates": MAX_REFERENCE_DATES,
    "max_window_days": MAX_WINDOW_DAYS,
    "window_step_days": WINDOW_STEP_DAYS,
    "coverage_target_pct": COVERAGE_TARGET_PCT,
    "area_scale_m": AREA_SCALE_M,
    "reduce_tile_scale": REDUCE_TILE_SCALE,
    "include_hls_sentinel2": INCLUDE_HLS_SENTINEL2,
    "aoi_area_km2": aoi_area_km2,
    "output_dir": str(OUTPUT_DIR),
}
parameters_json = OUTPUT_DIR / "parameters.json"
with parameters_json.open("w", encoding="utf-8") as f:
    json.dump(parameters, f, indent=2)

output_index = pd.DataFrame([
    {"file": str(CSV_DIR / "reference_windows.csv"), "description": "Reference dates and non-overlapping primary windows."},
    {"file": str(CSV_DIR / "window_scan_results.csv"), "description": "All target-date by window-day coverage results."},
    {"file": str(CSV_DIR / "fixed_primary_windows.csv"), "description": f"Coverage results for the fixed +/-{PRIMARY_WINDOW_DAYS} primary window."},
    {"file": str(CSV_DIR / "optimal_windows.csv"), "description": "Minimum window reaching the target, or best available window."},
    {"file": str(CSV_DIR / "reference_date_summary.csv"), "description": "Merged fixed-vs-optimal summary for review."},
    {"file": str(CSV_DIR / "coverage_curve_png_index.csv"), "description": "Index of one PNG coverage curve per reference date."},
    {"file": str(CSV_DIR / "figure_manifest.csv"), "description": "Index of summary PNG figures."},
    {"file": str(parameters_json), "description": "Run parameters."},
    {"file": str(OUTPUT_DIR / "aoi.geojson"), "description": "AOI used for the run."},
])
output_index_csv = CSV_DIR / "output_index.csv"
output_index.to_csv(output_index_csv, index=False)

print("Saved:", parameters_json)
print("Saved:", output_index_csv)
display(output_index)

Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\parameters.json
Saved: C:\Users\ibana\Desktop\SW_DWS1\Approaches\Approach3\notebooks\outputs\multidate_window_coverage_preanalysis\drawn_multidate_2025_20260702_140708\csv\output_index.csv


,file,description
0,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Reference dates and non-overlapping primary wi...
1,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,All target-date by window-day coverage results.
2,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Coverage results for the fixed +/-5 primary wi...
3,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,"Minimum window reaching the target, or best av..."
4,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Merged fixed-vs-optimal summary for review.
5,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Index of one PNG coverage curve per reference ...
6,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Index of summary PNG figures.
7,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,Run parameters.
8,C:\Users\ibana\Desktop\SW_DWS1\Approaches\Appr...,AOI used for the run.


## Takeaways

After running the notebook, start with these files:

- `csv/reference_date_summary.csv`: one row per reference date, comparing fixed `+/-5` coverage against the selected optimal window.
- `figures/summary_fixed_window_source_contribution.png`: the main plot for the non-overlapping processing cadence.
- `figures/summary_final_coverage_comparison.png`: where the fixed window falls below the target and whether the adaptive scan improves it.
- `figures/summary_optimal_window_days.png`: which reference dates need more than `+/-5` days.
- `figures/per_reference_curves/`: one coverage curve per reference date, equivalent to the single-date graph but saved as PNG.